# Blind Analytics — Pipeline Data Explorer

Notebook para explorar e validar os dados do pipeline de diagnóstico de mensuração.

**Fonte de dados:** `sample-diagnostic.json` (payload simulado)

**Seções:**
1. Carregamento dos dados
2. Distribuição de scores por módulo
3. Validação de regex patterns (identificação de tags)
4. Compliance de schema GA4 e-commerce
5. Análise de issues e recomendações

---
## 1. Carregamento dos Dados

In [ ]:
import json
import re
from pathlib import Path

# Carregar payload do diagnóstico
SAMPLE_PATH = Path("../examples/sample-diagnostic.json")
with open(SAMPLE_PATH) as f:
    diag = json.load(f)

print(f"Domínio: {diag['domain']}")
print(f"Timestamp: {diag['audit_timestamp']}")
print(f"Módulos avaliados: {len(diag['modules'])}")
print(f"Score geral: {diag['overall_maturity']['score']}/{diag['overall_maturity']['max_score']}")
print(f"Rating: {diag['overall_maturity']['rating']}")

---
## 2. Distribuição de Scores por Módulo

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extrair scores
modules = diag["modules"]
labels = {
    "tracking_infrastructure": "Infraestrutura\nde Tags",
    "attribution_health": "Saúde da\nAtribuição",
    "server_side_tracking": "Server-Side\nTracking",
    "datalayer_depth": "Profundidade\ndo DataLayer"
}

names = [labels[k] for k in modules]
scores = [modules[k]["score"] for k in modules]
max_scores = [modules[k]["max_score"] for k in modules]

# Cores por score
def score_color(s, m=5):
    pct = s / m
    if pct >= 0.8: return "#22c55e"
    if pct >= 0.6: return "#eab308"
    if pct >= 0.4: return "#f97316"
    return "#ef4444"

colors = [score_color(s) for s in scores]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax = axes[0]
bars = ax.bar(names, scores, color=colors, edgecolor="white", linewidth=1.5)
ax.set_ylim(0, 5.5)
ax.axhline(y=diag["overall_maturity"]["score"], color="#6366f1", linestyle="--", alpha=0.7, label=f"Média: {diag['overall_maturity']['score']}")
ax.set_ylabel("Score (0-5)")
ax.set_title("Score por Módulo", fontweight="bold")
ax.legend()
for bar, s in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15, str(s), ha="center", fontweight="bold", fontsize=12)

# Radar chart
ax = axes[1]
angles = np.linspace(0, 2 * np.pi, len(scores), endpoint=False).tolist()
scores_plot = scores + [scores[0]]
angles += [angles[0]]
short_names = ["Tags", "Atribuição", "SST", "DataLayer"]

ax = plt.subplot(122, polar=True)
ax.fill(angles, scores_plot, alpha=0.15, color="#6366f1")
ax.plot(angles, scores_plot, color="#6366f1", linewidth=2)
ax.set_ylim(0, 5)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(short_names, fontsize=10)
ax.set_title("Radar de Maturidade", fontweight="bold", pad=20)

plt.tight_layout()
plt.show()

---
## 3. Validação de Regex Patterns (Identificação de Tags)

Testa os regex patterns usados pelo pipeline para identificar tags em URLs de rede.

In [ ]:
# Regex patterns para identificação de tags (do protocolo)
TAG_PATTERNS = {
    "GTM": r"googletagmanager\.com/gtm\.js\?id=(GTM-[A-Z0-9]+)",
    "GA4": r"google-analytics\.com/g/collect\?.*?(?:tid|_p)=(G-[A-Z0-9]+)",
    "GA4 (gtag)": r"googletagmanager\.com/gtag/js\?id=(G-[A-Z0-9]+)",
    "Meta Pixel": r"connect\.facebook\.net/.*/fbevents\.js",
    "Meta Pixel ID": r"fbq\(['\"]init['\"],\s*['\"]([0-9]+)['\"]",
    "LinkedIn Insight": r"snap\.licdn\.com/li\.lms-analytics/insight\.min\.js",
}

# URLs simuladas de tráfego de rede (como seriam interceptadas)
SAMPLE_URLS = [
    "https://www.googletagmanager.com/gtm.js?id=GTM-ABC123",
    "https://www.googletagmanager.com/gtag/js?id=G-XYZ789",
    "https://www.google-analytics.com/g/collect?v=2&tid=G-XYZ789&_p=12345",
    "https://connect.facebook.net/en_US/fbevents.js",
    "https://www.googletagmanager.com/gtm.js?id=GTM-DUPLICATE",
    "https://cdn.example.com/script.js",
    "https://snap.licdn.com/li.lms-analytics/insight.min.js",
]

print("=" * 70)
print("VALIDAÇÃO DE REGEX PATTERNS")
print("=" * 70)

for url in SAMPLE_URLS:
    matches = []
    for tag_name, pattern in TAG_PATTERNS.items():
        match = re.search(pattern, url)
        if match:
            extracted_id = match.group(1) if match.lastindex else "(sem ID)"
            matches.append(f"{tag_name}: {extracted_id}")
    
    status = "\u2705" if matches else "\u2796"
    print(f"\n{status} {url[:80]}")
    for m in matches:
        print(f"   \u2192 {m}")

In [ ]:
# Teste de duplicatas
gtm_pattern = TAG_PATTERNS["GTM"]
gtm_ids = []
for url in SAMPLE_URLS:
    m = re.search(gtm_pattern, url)
    if m:
        gtm_ids.append(m.group(1))

print(f"\nGTM IDs encontrados: {gtm_ids}")
print(f"Duplicatas detectadas: {len(gtm_ids) != len(set(gtm_ids))}")
print(f"IDs únicos: {list(set(gtm_ids))}")

---
## 4. Compliance de Schema GA4 E-commerce

Valida se os eventos do DataLayer seguem o schema GA4 de e-commerce.

In [ ]:
# Schema GA4 E-commerce (baseado no Anexo A do protocolo)
GA4_ECOMMERCE_SCHEMA = {
    "required_events": [
        "view_item_list", "view_item", "add_to_cart", "view_cart",
        "begin_checkout", "add_payment_info", "add_shipping_info", "purchase"
    ],
    "required_item_fields": ["item_id", "item_name", "price", "quantity"],
    "recommended_item_fields": ["item_brand", "item_category", "item_category2", "item_variant", "discount", "coupon"],
    "required_purchase_fields": ["transaction_id", "value", "currency"]
}

# Dados do diagnóstico
dl_data = diag["modules"]["datalayer_depth"]["data"]
detected_events = dl_data["standard_events_detected"]
present_fields = dl_data["required_fields_present"]
missing_recommended = dl_data["missing_recommended_fields"]

print("=" * 70)
print("COMPLIANCE DE SCHEMA GA4 E-COMMERCE")
print("=" * 70)

# Cobertura de eventos
print("\n\u25b6 Cobertura de Eventos do Funil:")
for evt in GA4_ECOMMERCE_SCHEMA["required_events"]:
    status = "\u2705" if evt in detected_events else "\u274c"
    print(f"  {status} {evt}")

coverage = len([e for e in GA4_ECOMMERCE_SCHEMA["required_events"] if e in detected_events])
total = len(GA4_ECOMMERCE_SCHEMA["required_events"])
print(f"\n  Cobertura: {coverage}/{total} ({coverage/total*100:.0f}%)")

# Campos obrigatórios de items
print("\n\u25b6 Campos Obrigatórios do Array items[]:")
for field in GA4_ECOMMERCE_SCHEMA["required_item_fields"]:
    status = "\u2705" if field in present_fields else "\u274c"
    print(f"  {status} {field}")

# Campos recomendados
print("\n\u25b6 Campos Recomendados (ausentes):")
for field in missing_recommended:
    print(f"  \u26a0\ufe0f  {field}")

all_recommended = GA4_ECOMMERCE_SCHEMA["recommended_item_fields"]
present_recommended = [f for f in all_recommended if f not in missing_recommended]
print(f"\n  Recomendados presentes: {len(present_recommended)}/{len(all_recommended)}")

In [ ]:
# Visualização de compliance
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Eventos do funil
ax = axes[0]
present = len([e for e in GA4_ECOMMERCE_SCHEMA["required_events"] if e in detected_events])
missing = total - present
ax.pie([present, missing], labels=[f"Presentes ({present})", f"Ausentes ({missing})"],
       colors=["#22c55e", "#ef4444"], autopct="%1.0f%%", startangle=90)
ax.set_title("Eventos do Funil", fontweight="bold", fontsize=11)

# Campos obrigatórios
ax = axes[1]
req_present = len(present_fields)
req_total = len(GA4_ECOMMERCE_SCHEMA["required_item_fields"])
req_missing = req_total - req_present
ax.pie([req_present, req_missing], labels=[f"Presentes ({req_present})", f"Ausentes ({req_missing})"],
       colors=["#22c55e", "#ef4444"], autopct="%1.0f%%", startangle=90)
ax.set_title("Campos Obrigatórios", fontweight="bold", fontsize=11)

# Campos recomendados
ax = axes[2]
rec_present = len(present_recommended)
rec_missing = len(missing_recommended)
ax.pie([rec_present, rec_missing], labels=[f"Presentes ({rec_present})", f"Ausentes ({rec_missing})"],
       colors=["#22c55e", "#f97316"], autopct="%1.0f%%", startangle=90)
ax.set_title("Campos Recomendados", fontweight="bold", fontsize=11)

plt.suptitle("Compliance GA4 E-commerce Schema", fontweight="bold", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Análise de Issues e Recomendações

In [ ]:
# Issues por severidade
issues = diag["top_issues"]
severity_count = {}
for issue in issues:
    sev = issue["severity"]
    severity_count[sev] = severity_count.get(sev, 0) + 1

print("=" * 70)
print("RESUMO DE ISSUES")
print("=" * 70)
for sev, count in severity_count.items():
    emoji = "\U0001f534" if sev == "critical" else "\U0001f7e1"
    print(f"{emoji} {sev.upper()}: {count}")

print("\n" + "-" * 70)
for issue in issues:
    sev_label = "CRÍTICO" if issue["severity"] == "critical" else "ATENÇÃO"
    print(f"\n#{issue['rank']} [{sev_label}] {issue['title']}")
    print(f"   Módulo: {issue['module']}")
    print(f"   Impacto: {issue['business_impact']}")
    print(f"   Ação: {issue['recommendation']}")

In [ ]:
# Recomendações por nível de esforço
recs = diag["recommendations_summary"]

fig, ax = plt.subplots(figsize=(10, 4))

categories = ["Quick Wins", "Esforço Médio", "Esforço Alto"]
counts = [len(recs["quick_wins"]), len(recs["medium_effort"]), len(recs["high_effort"])]
colors = ["#22c55e", "#eab308", "#ef4444"]

bars = ax.barh(categories, counts, color=colors, edgecolor="white", height=0.5)
ax.set_xlabel("Quantidade de recomendações")
ax.set_title("Recomendações por Nível de Esforço", fontweight="bold")
ax.set_xlim(0, max(counts) + 1)

for bar, c in zip(bars, counts):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, str(c),
            va="center", fontweight="bold", fontsize=12)

plt.tight_layout()
plt.show()

# Detalhamento
print("\n\u2705 QUICK WINS:")
for r in recs["quick_wins"]:
    print(f"   \u2022 {r}")

print("\n\U0001f7e1 ESFORÇO MÉDIO:")
for r in recs["medium_effort"]:
    print(f"   \u2022 {r}")

print("\n\U0001f534 ESFORÇO ALTO:")
for r in recs["high_effort"]:
    print(f"   \u2022 {r}")

---
## Conclusão

Este notebook valida:

- **Contrato de dados**: O formato JSON do pipeline (`sample-diagnostic.json`) contém todos os campos necessários para alimentar o dashboard.
- **Regex patterns**: Os patterns identificam corretamente GTM, GA4, Meta Pixel e LinkedIn Insight em URLs de rede.
- **Schema GA4**: A validação detecta campos presentes e ausentes conforme o schema padrão de e-commerce.
- **Scoring**: A distribuição visual confirma que os scores refletem a maturidade real da configuração.

Próximos passos: integrar estes validadores nos helpers do pipeline (`tools/helpers/`).